In [2]:
import pandas as pd
import numpy as np 


In [3]:

import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "beer_reviews.csv"

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "thedevastator/1-5-million-beer-reviews-from-beer-advocate",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)


/Users/adyaverma/DS4420/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
brewdog = df[df['brewery_name'] == "BrewDog"]

In [5]:
brewdog

,index,brewery_id,brewery_name,review_time,review_overall,review_aroma,review_appearance,review_profilename,beer_style,review_palate,review_taste,beer_name,beer_abv,beer_beerid
738543,738543,16315,BrewDog,1270609465,4.0,4.0,4.0,northyorksammy,American Amber / Red Ale,3.5,3.5,5 A.M. Saint,5.0,55191
740545,740545,16315,BrewDog,1297822630,3.5,4.0,4.0,TheKingofWichita,American Amber / Red Ale,4.0,4.0,5 A.M. Saint,5.0,55191
740586,740586,16315,BrewDog,1320883160,3.5,3.5,3.5,cindyjeffsmith,American Double / Imperial IPA,3.5,4.0,Hardcore IPA (2nd Ed. 9.2%),9.2,64154
740595,740595,16315,BrewDog,1238813598,3.5,4.0,4.5,Chelsea1905,American Double / Imperial Stout,4.0,4.0,Rip Tide,8.0,41635
740652,740652,16315,BrewDog,1243432879,4.0,4.0,3.5,lordofthewiens,American IPA,4.0,4.5,Punk IPA,6.0,39435
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
750363,750363,16315,BrewDog,1308005120,4.0,4.0,4.0,MasterSki,American Double / Imperial IPA,4.0,4.0,"Hello, My Name Is Ingrid",8.2,67199
750364,750364,16315,BrewDog,1302634979,4.5,4.5,4.5,rarbring,American Double / Imperial IPA,4.5,4.5,"Hello, My Name Is Ingrid",8.2,67199
750365,750365,16315,BrewDog,1301685172,4.0,4.5,4.0,baggio,American Double / Imperial IPA,4.0,4.0,"Hello, My Name Is Ingrid",8.2,67199
750366,750366,16315,BrewDog,1301684783,4.5,4.5,4.0,bark,American Double / Imperial IPA,3.5,5.0,"Hello, My Name Is Ingrid",8.2,67199


In [6]:
import pandas as pd

recipes = pd.read_csv("/Users/adyaverma/Downloads/Table_S1.txt", sep="\t")
recipes.shape      
recipes.head()     
recipes.columns.tolist()

['Category.x',
 'recipe_code',
 'style',
 'BJCP',
 'hops_type',
 'alpha_acids_concentration',
 'IBU',
 'quantity_used',
 'hops_use_1',
 'hops_name_1',
 'Fermentation_type',
 'Region',
 'SG',
 'FG',
 'SG_converted',
 'bitterness_ratio',
 'AA_IBU_ratio',
 'FG_converted',
 'apparent_attenuation_percent',
 'ABV',
 'RBR',
 'time',
 'Fermentables',
 'quantities_kg',
 'Fermentable',
 'Category.y',
 'Type',
 'dist',
 'SRM',
 'batch_vol_liter',
 'mean_Color_Lovibond',
 'mcu',
 'efficiency_porcent',
 'Clusters',
 'Superclusters',
 'Phylum',
 'Class']

## Join reviews + recipes
The recipe table has no beer names (only anonymous `recipe_code`s), so the two datasets are joined on **style**: each BeerAdvocate style is mapped to a BJCP style, recipes are summarized per style (medians + most common hops/malts), and that profile is attached to every rated beer.

In [7]:

STYLE_MAP = {
    "Altbier": ("Altbier", "exact"),
    "American Adjunct Lager": ("American Lager", "exact"),
    "American Amber / Red Ale": ("American Amber Ale", "exact"),
    "American Amber / Red Lager": ("International Amber Lager", "approx"),
    "American Barleywine": ("American Barleywine", "exact"),
    "American Black Ale": ("Specialty IPA: Black IPA", "exact"),
    "American Blonde Ale": ("Blonde Ale", "exact"),
    "American Brown Ale": ("American Brown Ale", "exact"),
    "American Dark Wheat Ale": ("American Wheat Beer", "approx"),
    "American Double / Imperial IPA": ("Double IPA", "exact"),
    "American Double / Imperial Pilsner": ("German Pils", "approx"),
    "American Double / Imperial Stout": ("Imperial Stout", "exact"),
    "American IPA": ("American IPA", "exact"),
    "American Malt Liquor": ("American Lager", "approx"),
    "American Pale Ale (APA)": ("American Pale Ale", "exact"),
    "American Pale Lager": ("International Pale Lager", "approx"),
    "American Pale Wheat Ale": ("American Wheat Beer", "exact"),
    "American Porter": ("American Porter", "exact"),
    "American Stout": ("American Stout", "exact"),
    "American Strong Ale": ("American Strong Ale", "exact"),
    "American Wild Ale": ("Wild Specialty Beer", "approx"),
    "Baltic Porter": ("Baltic Porter", "exact"),
    "Belgian Dark Ale": ("Belgian Dubbel", "approx"),
    "Belgian IPA": ("Specialty IPA: Belgian IPA", "exact"),
    "Belgian Pale Ale": ("Belgian Pale Ale", "exact"),
    "Belgian Strong Dark Ale": ("Belgian Dark Strong Ale", "exact"),
    "Belgian Strong Pale Ale": ("Belgian Golden Strong Ale", "exact"),
    "Berliner Weissbier": ("Berliner Weisse", "exact"),
    "Bière de Garde": ("Bière de Garde", "exact"),
    "Bock": ("Dunkles Bock", "exact"),
    "California Common / Steam Beer": ("California Common", "exact"),
    "Chile Beer": ("Spice, Herb, or Vegetable Beer", "approx"),
    "Cream Ale": ("Cream Ale", "exact"),
    "Czech Pilsener": ("Czech Premium Pale Lager", "exact"),
    "Doppelbock": ("Doppelbock", "exact"),
    "Dortmunder / Export Lager": ("German Helles Exportbier", "exact"),
    "Dubbel": ("Belgian Dubbel", "exact"),
    "Dunkelweizen": ("Dunkles Weissbier", "exact"),
    "Eisbock": ("Eisbock", "exact"),
    "English Barleywine": ("English Barleywine", "exact"),
    "English Bitter": ("Best Bitter", "exact"),
    "English Brown Ale": ("British Brown Ale", "exact"),
    "English Dark Mild Ale": ("Dark Mild", "exact"),
    "English India Pale Ale (IPA)": ("English IPA", "exact"),
    "English Pale Ale": ("Best Bitter", "approx"),
    "English Pale Mild Ale": ("Ordinary Bitter", "approx"),
    "English Porter": ("English Porter", "exact"),
    "English Stout": ("Irish Extra Stout", "approx"),
    "English Strong Ale": ("British Strong Ale", "exact"),
    "Euro Dark Lager": ("International Dark Lager", "exact"),
    "Euro Pale Lager": ("International Pale Lager", "exact"),
    "Euro Strong Lager": ("International Pale Lager", "approx"),
    "Extra Special / Strong Bitter (ESB)": ("Strong Bitter", "exact"),
    "Faro": ("Lambic", "approx"),
    "Flanders Oud Bruin": ("Oud Bruin", "exact"),
    "Flanders Red Ale": ("Flanders Red Ale", "exact"),
    "Foreign / Export Stout": ("Foreign Extra Stout", "exact"),
    "Fruit / Vegetable Beer": ("Fruit Beer", "exact"),
    "German Pilsener": ("German Pils", "exact"),
    "Gose": ("Gose", "exact"),
    "Gueuze": ("Gueuze", "exact"),
    "Hefeweizen": ("Weissbier", "exact"),
    "Herbed / Spiced Beer": ("Spice, Herb, or Vegetable Beer", "exact"),
    "Irish Dry Stout": ("Irish Stout", "exact"),
    "Irish Red Ale": ("Irish Red Ale", "exact"),
    "Japanese Rice Lager": ("International Pale Lager", "approx"),
    "Keller Bier / Zwickel Bier": ("Kellerbier: Pale Kellerbier", "exact"),
    "Kristalweizen": ("Weissbier", "approx"),
    "Kölsch": ("Kölsch", "exact"),
    "Lambic - Fruit": ("Fruit Lambic", "exact"),
    "Lambic - Unblended": ("Lambic", "exact"),
    "Light Lager": ("American Light Lager", "exact"),
    "Low Alcohol Beer": ("German Leichtbier", "approx"),
    "Maibock / Helles Bock": ("Helles Bock", "exact"),
    "Milk / Sweet Stout": ("Sweet Stout", "exact"),
    "Munich Dunkel Lager": ("Munich Dunkel", "exact"),
    "Munich Helles Lager": ("Munich Helles", "exact"),
    "Märzen / Oktoberfest": ("Märzen", "exact"),
    "Oatmeal Stout": ("Oatmeal Stout", "exact"),
    "Old Ale": ("Old Ale", "exact"),
    "Pumpkin Ale": ("Autumn Seasonal Beer", "approx"),
    "Quadrupel (Quad)": ("Belgian Dark Strong Ale", "approx"),
    "Rauchbier": ("Rauchbier", "exact"),
    "Roggenbier": ("Roggenbier", "exact"),
    "Russian Imperial Stout": ("Imperial Stout", "exact"),
    "Rye Beer": ("Alternative Grain Beer", "approx"),
    "Sahti": ("Sahti", "exact"),
    "Saison / Farmhouse Ale": ("Saison", "exact"),
    "Schwarzbier": ("Schwarzbier", "exact"),
    "Scotch Ale / Wee Heavy": ("Wee Heavy", "exact"),
    "Scottish Ale": ("Scottish Export", "approx"),
    "Scottish Gruit / Ancient Herbed Ale": ("Spice, Herb, or Vegetable Beer", "approx"),
    "Smoked Beer": ("Specialty Smoked Beer", "exact"),
    "Tripel": ("Belgian Tripel", "exact"),
    "Vienna Lager": ("Vienna Lager", "exact"),
    "Weizenbock": ("Weizenbock", "exact"),
    "Wheatwine": ("Wheatwine", "exact"),
    "Winter Warmer": ("Winter Seasonal Beer", "approx"),
    "Witbier": ("Witbier", "exact"),

}
style_map = pd.DataFrame(
    [(ba, bj, q) for ba, (bj, q) in STYLE_MAP.items()],
    columns=["beer_style", "recipe_style", "style_match"],
)
assert style_map.recipe_style.isin(recipes["style"]).all()

#keep only one row per recipe
r = recipes.copy()
r["_tmin"] = r.groupby("recipe_code")["time"].transform("min")
hop_rows = r[r["time"] == r["_tmin"]]                       
ferm_rows = r.drop_duplicates(["recipe_code", "time"])      

per_recipe = r.drop_duplicates("recipe_code").set_index("recipe_code")[
    ["style", "Fermentation_type", "Phylum", "Class",
     "SG", "FG", "ABV", "SRM", "apparent_attenuation_percent", "efficiency_porcent"]
]
per_recipe["IBU"] = hop_rows.groupby("recipe_code")["IBU"].sum()
per_recipe["hop_g_per_L"] = (
    hop_rows.groupby("recipe_code")["quantity_used"].sum()
    / r.drop_duplicates("recipe_code").set_index("recipe_code")["batch_vol_liter"]
)
per_recipe["dry_hop_frac"] = (
    hop_rows.assign(d=hop_rows["hops_use_1"].eq("Dry Hop") * hop_rows["quantity_used"])
    .groupby("recipe_code")[["d", "quantity_used"]].sum()
    .pipe(lambda x: x["d"] / x["quantity_used"])
)
# Grain bill composition (% of fermentable kg by malt type)
bill = ferm_rows.pivot_table(index="recipe_code", columns="Type",
                             values="quantities_kg", aggfunc="sum", fill_value=0)
bill = bill.div(bill.sum(axis=1), axis=0).mul(100)
bill_cols = {"Base malt": "pct_base_malt", "Crystal malt": "pct_crystal_malt",
             "Roasted malt": "pct_roasted_malt", "Specialty malt": "pct_specialty_malt",
             "Raw": "pct_raw_grain", "Sugar": "pct_sugar"}
per_recipe = per_recipe.join(bill[list(bill_cols)].rename(columns=bill_cols))

# --- 3. Recipe profile per style ---
num_cols = ["SG", "FG", "ABV", "IBU", "SRM", "apparent_attenuation_percent",
            "efficiency_porcent", "hop_g_per_L", "dry_hop_frac", *bill_cols.values()]
mode = lambda s: s.mode().iat[0]
top_n = lambda s, n=5: ", ".join(s.value_counts().head(n).index)

g = per_recipe.groupby("style")
style_profile = g[num_cols].median().add_prefix("recipe_median_")
style_profile.insert(0, "n_recipes", g.size())
style_profile["recipe_fermentation"] = g["Fermentation_type"].agg(mode)
style_profile["recipe_phylum"] = g["Phylum"].agg(mode)
style_profile["recipe_class"] = g["Class"].agg(mode)
# Most common hops / fermentables (counted once per recipe)
hops_by_style = hop_rows.drop_duplicates(["recipe_code", "hops_name_1"]).groupby("style")["hops_name_1"]
ferm_by_style = ferm_rows.drop_duplicates(["recipe_code", "Fermentable"]).groupby("style")["Fermentable"]
style_profile["top_hops"] = hops_by_style.agg(top_n)
style_profile["top_fermentables"] = ferm_by_style.agg(top_n)
style_profile = style_profile.reset_index().rename(columns={"style": "recipe_style"})

# --- 4. One row per rated beer ---
score_cols = ["review_overall", "review_aroma", "review_appearance", "review_palate", "review_taste"]
beers = (
    df.groupby("beer_beerid")
    .agg(beer_name=("beer_name", "first"), brewery_name=("brewery_name", "first"),
         beer_style=("beer_style", "first"), beer_abv=("beer_abv", "first"),
         n_reviews=("review_overall", "size"),
         **{f"avg_{c.removeprefix('review_')}": (c, "mean") for c in score_cols})
    .reset_index()
)

# --- 5. Join ---
beer_recipes = (
    beers.merge(style_map, on="beer_style", how="inner")
         .merge(style_profile, on="recipe_style", how="left")
)
print(f"{len(beer_recipes):,} of {len(beers):,} beers matched "
      f"({beer_recipes.n_reviews.sum():,} reviews); "
      f"{(beer_recipes.style_match == 'exact').mean():.0%} via exact style match")
beer_recipes.to_csv("beer_recipes_ratings.csv", index=False)


65,777 of 66,055 beers matched (1,581,632 reviews); 89% via exact style match


In [8]:
beer_recipes.head()

,beer_beerid,beer_name,brewery_name,beer_style,beer_abv,n_reviews,avg_overall,avg_aroma,avg_appearance,avg_palate,...,recipe_median_pct_crystal_malt,recipe_median_pct_roasted_malt,recipe_median_pct_specialty_malt,recipe_median_pct_raw_grain,recipe_median_pct_sugar,recipe_fermentation,recipe_phylum,recipe_class,top_hops,top_fermentables
0,3,Cactus Queen IPA,Yellow Rose Brewing Company,American IPA,NaN,3,4.166667,4.000000,3.833333,4.166667,...,5.263158,0.000000,0.0,0.000000,0.0,Hot fermented,Hop-Forward and Strong Ales,New IPAs,"Citra, Cascade, Centennial, Amarillo, Simcoe","Pale 2-Row, Carapils, Maris Otter Pale, Flaked..."
1,4,Wildcatter's Crude Stout,Yellow Rose Brewing Company,American Stout,NaN,10,3.700000,3.800000,3.950000,3.700000,...,5.610457,11.764706,0.0,4.259214,0.0,Hot fermented,Malty and Roasted Ales,Stouts and Porters,"Goldings, Cascade, Magnum, Willamette, Fuggles","Chocolate, Roasted Barley, Pale 2-Row, Flaked ..."
2,5,Amber,Abita Brewing Co.,Vienna Lager,4.5,424,3.549528,3.205189,3.485849,3.318396,...,3.344482,0.334448,0.0,0.000000,0.0,Cold fermented,Cold Fermented Classics,Amber Lagers,"Tettnanger, Saaz, Hallertau Mittelfrüh, Magnum...","Vienna, Pilsner, Carapils, Melanoidin, Caramel..."
3,6,Turbodog,Abita Brewing Co.,English Brown Ale,5.6,877,3.706956,3.515964,3.836374,3.513113,...,7.692308,6.349206,0.0,0.000000,0.0,Hot fermented,Malty and Roasted Ales,Brown Ales,"Goldings, Fuggles, Cascade, Willamette, Magnum","Maris Otter Pale, Chocolate, Pale 2-Row, Pale ..."
4,7,Purple Haze,Abita Brewing Co.,Fruit / Vegetable Beer,4.2,659,3.266313,3.179059,3.251138,3.106980,...,0.000000,0.000000,0.0,0.000000,0.0,Hot fermented,Diverse Fermentation Profiles,Alternative and Specialty Grain Beers,"Cascade, Citra, Saaz, Magnum, Goldings","Pale 2-Row, Pilsner, Wheat, White Wheat, Carapils"
